# CIFAR-10 Image Colorization Conditional VAE V2

It uses the same preprocessed CIFAR-10 HDF5 files as the CNN and GAN models:

- **Input:** normalized CIE-LAB \(L\) channel
- **Target:** normalized CIE-LAB \(ab\) channels
- **Train:** 45,000 images
- **Validation:** 5,000 images
- **Test:** 10,000 images


The posterior is

\[
q_\phi(z\mid L,ab)
\]

and the conditional prior is

\[
p_\psi(z\mid L).
\]

The decoder predicts

\[
\widehat{ab}=D_\theta(L,z).
\]

The training objective is

\[
\mathcal L =
\mathcal L_{\text{weighted-L1}}
+
\lambda_c \mathcal L_{\text{chroma}}
+
\beta D_{KL}\left(q_\phi(z\mid L,ab)\,\Vert\,p_\psi(z\mid L)\right).
\]

## Final comparison metrics

The notebook reports the  four metrics:

- **Normalized `ab` MSE **
- **Normalized `ab` MAE **
- **RGB PSNR **
- **RGB SSIM **

It also saves everything to Google Drive.

## 1. Install packages

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "h5py", "scikit-image", "scikit-learn",
    "pandas", "matplotlib", "tqdm"
])

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Imports, reproducibility, and GPU

In [ ]:
import gc
import json
import math
import random
import shutil
import time
from collections import defaultdict
from contextlib import nullcontext
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from IPython.display import display
from sklearn.decomposition import PCA
from skimage.color import lab2rgb
from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity,
)
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

print("PyTorch:", torch.__version__)
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
        "GB",
    )

assert device.type == "cuda", (
    "This notebook is intended for Colab GPU training. "
    "Enable GPU under Runtime > Change runtime type."
)

## 4. Locate the shared CIFAR-10 HDF5 data

The first candidate below matches the Drive location used by the completed V1 run.  
The notebook also checks the earlier project folder structure automatically.

In [ ]:
DRIVE_CANDIDATES = [
    Path("/content/drive/MyDrive/ECE1508Data/cifar10_lab_v1"),
    Path(
        "/content/drive/MyDrive/"
        "ECE1508_Image_Colorization/shared_data/cifar10_lab_v1"
    ),
    Path(
        "/content/drive/MyDrive/"
        "ECE1508_Image_Colorization/cifar10_lab_v1"
    ),
]

required_names = [
    "train_lab.h5",
    "validation_lab.h5",
    "test_lab.h5",
]

DRIVE_DATA_DIR = None

for candidate in DRIVE_CANDIDATES:
    if all((candidate / name).exists() for name in required_names):
        DRIVE_DATA_DIR = candidate
        break

if DRIVE_DATA_DIR is None:
    print("Checked:")
    for candidate in DRIVE_CANDIDATES:
        print(" -", candidate)

    raise FileNotFoundError(
        "Could not locate the shared CIFAR-10 HDF5 files. "
        "Edit DRIVE_CANDIDATES in this cell if your folder is elsewhere."
    )

print("Shared dataset:", DRIVE_DATA_DIR)

for name in required_names:
    path = DRIVE_DATA_DIR / name
    print(f"{name:22s} {path.stat().st_size / 1024**2:8.1f} MB")

## 5. Copy data to local storage


In [ ]:
COPY_DATA_TO_LOCAL = True

LOCAL_DATA_DIR = Path("/content/cifar10_lab_v1")

if COPY_DATA_TO_LOCAL:
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

    for name in required_names:
        source = DRIVE_DATA_DIR / name
        destination = LOCAL_DATA_DIR / name

        if (
            not destination.exists()
            or destination.stat().st_size != source.stat().st_size
        ):
            print("Copying:", name)
            shutil.copy2(source, destination)
        else:
            print("Already copied:", name)

    DATA_DIR = LOCAL_DATA_DIR
else:
    DATA_DIR = DRIVE_DATA_DIR

DRIVE_OUTPUT_DIR = (
    DRIVE_DATA_DIR
    / "results"
    / "cvae_v2_full"
)

MODEL_DIR = DRIVE_OUTPUT_DIR / "models"
FIGURE_DIR = DRIVE_OUTPUT_DIR / "figures"
METRIC_DIR = DRIVE_OUTPUT_DIR / "metrics"

for folder in [MODEL_DIR, FIGURE_DIR, METRIC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Training data:", DATA_DIR)
print("Drive results:", DRIVE_OUTPUT_DIR)

## 6. Configuration

In [ ]:
BATCH_SIZE = 512
NUM_WORKERS = 8
LATENT_DIM = 64

EPOCHS = 30
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-5

BETA_MAX = 0.005
KL_WARMUP_EPOCHS = 30

# Chroma-aware loss.
CHROMA_WEIGHT_ALPHA = 2.0
CHROMA_SCALE = 0.35
CHROMA_AUX_WEIGHT = 0.20

EARLY_STOPPING_PATIENCE = 7
CHECKPOINT_EVERY = 5

USE_AMP = True
AMP_DTYPE = torch.bfloat16
RESUME_TRAINING = True

# None = evaluate all 10,000 test images.
FINAL_METRIC_IMAGES = None

CIFAR10_CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

config = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "latent_dim": LATENT_DIM,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "beta_max": BETA_MAX,
    "kl_warmup_epochs": KL_WARMUP_EPOCHS,
    "chroma_weight_alpha": CHROMA_WEIGHT_ALPHA,
    "chroma_scale": CHROMA_SCALE,
    "chroma_aux_weight": CHROMA_AUX_WEIGHT,
    "seed": SEED,
}

with open(METRIC_DIR / "v2_config.json", "w") as file:
    json.dump(config, file, indent=2)

print(json.dumps(config, indent=2))


This run uses **batch size 512**, **8 DataLoader workers**, **BF16 mixed precision**, `channels_last`, cuDNN benchmarking, TF32 for eligible FP32 operations, pinned memory, non-blocking transfers, and local `/content` HDF5 files. These settings prioritize training speed without changing the VAE objective.

In [ ]:
print("Batch size:", BATCH_SIZE)
print("Workers:", NUM_WORKERS)
print("AMP dtype:", AMP_DTYPE)
print("cuDNN benchmark:", torch.backends.cudnn.benchmark)
print("TF32 matmul:", torch.backends.cuda.matmul.allow_tf32)
print("TF32 cuDNN:", torch.backends.cudnn.allow_tf32)

## 7. HDF5 dataset loader

In [ ]:
class LABH5Dataset(Dataset):
    def __init__(self, path):
        self.path = str(path)
        self._file = None

        with h5py.File(self.path, "r") as file:
            self.length = len(file["labels"])

    def _open(self):
        if self._file is None:
            self._file = h5py.File(self.path, "r")

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        self._open()

        L = torch.from_numpy(
            self._file["L"][index].astype(np.float32)
        )
        ab = torch.from_numpy(
            self._file["ab"][index].astype(np.float32)
        )
        label = int(self._file["labels"][index])
        source_index = int(self._file["source_indices"][index])

        return L, ab, label, source_index

    def close(self):
        if self._file is not None:
            self._file.close()
            self._file = None

    def __del__(self):
        self.close()


train_dataset = LABH5Dataset(DATA_DIR / "train_lab.h5")
val_dataset = LABH5Dataset(DATA_DIR / "validation_lab.h5")
test_dataset = LABH5Dataset(DATA_DIR / "test_lab.h5")

common_loader_args = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": True,
    "persistent_workers": NUM_WORKERS > 0,
    "prefetch_factor": 4 if NUM_WORKERS > 0 else None,
}

if NUM_WORKERS == 0:
    common_loader_args.pop("prefetch_factor", None)

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    **common_loader_args,
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **common_loader_args,
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **common_loader_args,
)

print(f"Train:      {len(train_dataset):,}")
print(f"Validation: {len(val_dataset):,}")
print(f"Test:       {len(test_dataset):,}")

## 8. Verify data and save dataset examples

In [ ]:
def normalized_L_to_gray(L):
    array = L.detach().cpu().squeeze(0).numpy().astype(np.float32)
    return np.clip((array + 1.0) / 2.0, 0.0, 1.0)


def normalized_lab_to_rgb(L, ab):
    L_hwc = (
        L.detach().cpu()
        .permute(1, 2, 0)
        .numpy()
        .astype(np.float32)
    )

    ab_hwc = (
        ab.detach().cpu()
        .permute(1, 2, 0)
        .numpy()
        .astype(np.float32)
    )

    L_real = (L_hwc + 1.0) * 50.0
    ab_real = ab_hwc * 128.0

    lab = np.concatenate([L_real, ab_real], axis=-1)
    return np.clip(lab2rgb(lab), 0.0, 1.0)


L_batch, ab_batch, labels_batch, _ = next(iter(train_loader))

print("L:", tuple(L_batch.shape), L_batch.dtype)
print("ab:", tuple(ab_batch.shape), ab_batch.dtype)
print("L range:", float(L_batch.min()), float(L_batch.max()))
print("ab range:", float(ab_batch.min()), float(ab_batch.max()))

assert L_batch.shape[1:] == (1, 32, 32)
assert ab_batch.shape[1:] == (2, 32, 32)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))

for index in range(8):
    axes[0, index].imshow(
        normalized_L_to_gray(L_batch[index]),
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[0, index].axis("off")

    axes[1, index].imshow(
        normalized_lab_to_rgb(
            L_batch[index],
            ab_batch[index],
        )
    )
    axes[1, index].axis("off")

axes[0, 0].set_ylabel("Grayscale")
axes[1, 0].set_ylabel("Ground truth")
fig.suptitle("Shared CIFAR-10 LAB dataset")
plt.tight_layout()

fig.savefig(
    FIGURE_DIR / "01_dataset_examples.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()

## 9. V1 baseline architecture

This class is included only so the notebook can load the checkpoint produced by the completed V1 notebook and calculate the **same four group metrics** for it.

V1 generation uses \(z=0\), matching the completed notebook.

In [ ]:
class CVAEV1(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim

        self.posterior_encoder = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)

        self.condition_encoder = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        self.z_projection = nn.Linear(
            latent_dim,
            256 * 4 * 4,
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 2, 3, 1, 1),
            nn.Tanh(),
        )

    def decode(self, L, z):
        condition = self.condition_encoder(L)

        z_features = (
            self.z_projection(z)
            .view(z.size(0), 256, 4, 4)
        )

        return self.decoder(
            torch.cat(
                [condition, z_features],
                dim=1,
            )
        )

## 10. V2 conditional-prior CVAE

V2 learns a prior from the grayscale image:

\[
p_\psi(z\mid L)=
\mathcal N\left(
\mu_p(L),
\operatorname{diag}(\sigma_p^2(L))
\right).
\]

During training, the posterior is

\[
q_\phi(z\mid L,ab).
\]

At test time, the ground-truth colour is unavailable, so deterministic prediction uses the **conditional prior mean** \(z=\mu_p(L)\). Random samples are drawn from the same conditional prior.

In [ ]:
class CVAEV2(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim

        # ---------------------------
        # Posterior q(z | L, ab)
        # ---------------------------
        self.q1 = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.q2 = nn.Sequential(
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.q3 = nn.Sequential(
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.q_mu = nn.Linear(
            256 * 4 * 4,
            latent_dim,
        )
        self.q_logvar = nn.Linear(
            256 * 4 * 4,
            latent_dim,
        )

        # ---------------------------
        # Conditional prior p(z | L)
        # These features also become decoder skips.
        # ---------------------------
        self.p1 = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.p2 = nn.Sequential(
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.p3 = nn.Sequential(
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        self.p_mu = nn.Linear(
            256 * 4 * 4,
            latent_dim,
        )
        self.p_logvar = nn.Linear(
            256 * 4 * 4,
            latent_dim,
        )

        # ---------------------------
        # Decoder
        # ---------------------------
        self.z_projection = nn.Linear(
            latent_dim,
            256 * 4 * 4,
        )

        # [p3, z] : 512 x 4 x 4 -> 256 x 8 x 8
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(
                512,
                256,
                4,
                2,
                1,
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        # [up1, p2] : 384 x 8 x 8 -> 128 x 16 x 16
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(
                384,
                128,
                4,
                2,
                1,
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        # [up2, p1] : 192 x 16 x 16 -> 64 x 32 x 32
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(
                192,
                64,
                4,
                2,
                1,
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        # Include original L at the final resolution.
        self.out = nn.Sequential(
            nn.Conv2d(65, 64, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 2, 3, 1, 1),
            nn.Tanh(),
        )

    def encode_posterior(self, L, ab):
        x = torch.cat([L, ab], dim=1)

        h = self.q1(x)
        h = self.q2(h)
        h = self.q3(h)
        h = h.flatten(start_dim=1)

        mu = self.q_mu(h)
        logvar = torch.clamp(
            self.q_logvar(h),
            -10.0,
            10.0,
        )

        return mu, logvar

    def encode_prior(self, L):
        f1 = self.p1(L)
        f2 = self.p2(f1)
        f3 = self.p3(f2)

        flat = f3.flatten(start_dim=1)

        mu = self.p_mu(flat)
        logvar = torch.clamp(
            self.p_logvar(flat),
            -10.0,
            10.0,
        )

        return mu, logvar, (f1, f2, f3)

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)

        return (
            mu
            + std
            * torch.randn_like(std)
        )

    def decode_from_features(
        self,
        L,
        z,
        condition_features,
    ):
        f1, f2, f3 = condition_features

        z_features = (
            self.z_projection(z)
            .view(z.size(0), 256, 4, 4)
        )

        h = self.up1(
            torch.cat(
                [f3, z_features],
                dim=1,
            )
        )

        h = self.up2(
            torch.cat(
                [h, f2],
                dim=1,
            )
        )

        h = self.up3(
            torch.cat(
                [h, f1],
                dim=1,
            )
        )

        return self.out(
            torch.cat(
                [h, L],
                dim=1,
            )
        )

    def decode_prior(
        self,
        L,
        sample=False,
        temperature=1.0,
    ):
        mu_p, logvar_p, features = self.encode_prior(L)

        if sample:
            std = torch.exp(0.5 * logvar_p)

            z = (
                mu_p
                + temperature
                * std
                * torch.randn_like(std)
            )
        else:
            z = mu_p

        pred_ab = self.decode_from_features(
            L,
            z,
            features,
        )

        return pred_ab, mu_p, logvar_p

    def forward(self, L, ab):
        mu_q, logvar_q = self.encode_posterior(
            L,
            ab,
        )

        mu_p, logvar_p, features = self.encode_prior(L)

        z = self.reparameterize(
            mu_q,
            logvar_q,
        )

        pred_ab = self.decode_from_features(
            L,
            z,
            features,
        )

        return (
            pred_ab,
            mu_q,
            logvar_q,
            mu_p,
            logvar_p,
        )


model = CVAEV2(
    latent_dim=LATENT_DIM
).to(device).to(memory_format=torch.channels_last)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("V2 trainable parameters:", f"{parameter_count:,}")

## 11. Architecture diagram for the presentation

In [ ]:
fig, axis = plt.subplots(figsize=(13, 5))
axis.axis("off")

boxes = [
    (0.05, 0.68, "Grayscale L\n1×32×32"),
    (0.27, 0.78, "Conditional prior\np(z | L)"),
    (0.27, 0.35, "Posterior\nq(z | L, ab)"),
    (0.51, 0.57, "Latent z\n64-D"),
    (0.70, 0.57, "Skip-conditioned\ndecoder"),
    (0.90, 0.57, "Predicted ab\n2×32×32"),
]

for x, y, text in boxes:
    axis.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=12,
        bbox=dict(
            boxstyle="round,pad=0.5",
            facecolor="white",
            edgecolor="black",
        ),
        transform=axis.transAxes,
    )

axis.text(
    0.05,
    0.32,
    "True ab\n(training only)",
    ha="center",
    va="center",
    fontsize=12,
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        edgecolor="black",
    ),
    transform=axis.transAxes,
)

arrows = [
    ((0.10, 0.68), (0.22, 0.78)),
    ((0.10, 0.62), (0.22, 0.39)),
    ((0.10, 0.32), (0.22, 0.35)),
    ((0.34, 0.78), (0.46, 0.62)),
    ((0.34, 0.35), (0.46, 0.53)),
    ((0.56, 0.57), (0.65, 0.57)),
    ((0.76, 0.57), (0.85, 0.57)),
    ((0.10, 0.70), (0.65, 0.62)),
]

for start, end in arrows:
    axis.annotate(
        "",
        xy=end,
        xytext=start,
        xycoords=axis.transAxes,
        textcoords=axis.transAxes,
        arrowprops=dict(
            arrowstyle="->",
            linewidth=1.5,
        ),
    )

axis.set_title(
    "Conditional VAE V2 for Grayscale-to-Color Generation",
    fontsize=16,
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR / "02_cvae_v2_architecture.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()

## 12. Shape smoke test

In [ ]:
model.eval()

with torch.no_grad():
    sample_L = L_batch[:4].to(device)
    sample_ab = ab_batch[:4].to(device)

    (
        pred_ab,
        mu_q,
        logvar_q,
        mu_p,
        logvar_p,
    ) = model(
        sample_L,
        sample_ab,
    )

    prior_pred, _, _ = model.decode_prior(
        sample_L,
        sample=False,
    )

print("L:", tuple(sample_L.shape))
print("ab:", tuple(sample_ab.shape))
print("posterior prediction:", tuple(pred_ab.shape))
print("prior-mean prediction:", tuple(prior_pred.shape))
print("mu_q:", tuple(mu_q.shape))
print("mu_p:", tuple(mu_p.shape))

assert pred_ab.shape == sample_ab.shape
assert prior_pred.shape == sample_ab.shape
assert mu_q.shape == (4, LATENT_DIM)
assert mu_p.shape == (4, LATENT_DIM)

print("V2 smoke test passed.")

## 13. Chroma-aware reconstruction and conditional KL

The target chroma magnitude is

\[
c=\sqrt{a^2+b^2}.
\]

Pixels with more chroma receive more reconstruction weight. This directly addresses the tendency of an L1 colorizer to prefer a low-saturation average.

The KL term between two diagonal Gaussians is calculated as

\[
D_{KL}(q\Vert p)
=
\frac12
\sum_j
\left[
\log\frac{\sigma_{p,j}^2}{\sigma_{q,j}^2}
+
\frac{
\sigma_{q,j}^2+
(\mu_{q,j}-\mu_{p,j})^2
}{
\sigma_{p,j}^2
}
-1
\right].
\]

In [ ]:
def conditional_kl(
    mu_q,
    logvar_q,
    mu_p,
    logvar_p,
):
    var_q = torch.exp(logvar_q)
    var_p = torch.exp(logvar_p)

    kl_per_dim = 0.5 * (
        logvar_p
        - logvar_q
        + (
            var_q
            + (mu_q - mu_p).pow(2)
        )
        / var_p
        - 1.0
    )

    # Average per image and latent dimension.
    return kl_per_dim.mean()


def reconstruction_terms(
    pred_ab,
    true_ab,
):
    true_chroma = torch.sqrt(
        true_ab[:, 0:1].pow(2)
        + true_ab[:, 1:2].pow(2)
        + 1e-8
    )

    pred_chroma = torch.sqrt(
        pred_ab[:, 0:1].pow(2)
        + pred_ab[:, 1:2].pow(2)
        + 1e-8
    )

    color_strength = torch.clamp(
        true_chroma / CHROMA_SCALE,
        0.0,
        1.0,
    )

    pixel_weights = (
        1.0
        + CHROMA_WEIGHT_ALPHA
        * color_strength
    )

    weighted_l1 = (
        (
            torch.abs(
                pred_ab - true_ab
            )
            * pixel_weights
        ).mean()
    )

    chroma_loss = F.l1_loss(
        pred_chroma,
        true_chroma,
        reduction="mean",
    )

    plain_l1 = F.l1_loss(
        pred_ab,
        true_ab,
        reduction="mean",
    )

    return (
        weighted_l1,
        chroma_loss,
        plain_l1,
    )


def cvae_v2_loss(
    pred_ab,
    true_ab,
    mu_q,
    logvar_q,
    mu_p,
    logvar_p,
    beta,
):
    (
        weighted_l1,
        chroma_loss,
        plain_l1,
    ) = reconstruction_terms(
        pred_ab,
        true_ab,
    )

    kl = conditional_kl(
        mu_q,
        logvar_q,
        mu_p,
        logvar_p,
    )

    reconstruction_objective = (
        weighted_l1
        + CHROMA_AUX_WEIGHT
        * chroma_loss
    )

    total = (
        reconstruction_objective
        + beta
        * kl
    )

    # Fixed score for checkpoint comparison across all epochs.
    # Unlike the training loss, this always uses BETA_MAX.
    fixed_selection_score = (
        reconstruction_objective
        + BETA_MAX
        * kl
    )

    return {
        "total": total,
        "weighted_l1": weighted_l1,
        "plain_l1": plain_l1,
        "chroma": chroma_loss,
        "kl": kl,
        "selection": fixed_selection_score,
    }


def beta_for_epoch(epoch_index):
    if KL_WARMUP_EPOCHS <= 0:
        return BETA_MAX

    fraction = min(
        (epoch_index + 1)
        / KL_WARMUP_EPOCHS,
        1.0,
    )

    return BETA_MAX * fraction


for epoch_index in [0, 4, 9, 19, 39]:
    print(
        f"Epoch {epoch_index + 1:02d}: "
        f"beta={beta_for_epoch(epoch_index):.6f}"
    )

## 14. Initial loss check

In [ ]:
model.train()

sample_L = L_batch[:16].to(device)
sample_ab = ab_batch[:16].to(device)

outputs = model(
    sample_L,
    sample_ab,
)

losses = cvae_v2_loss(
    outputs[0],
    sample_ab,
    outputs[1],
    outputs[2],
    outputs[3],
    outputs[4],
    beta_for_epoch(0),
)

for key, value in losses.items():
    print(f"{key:12s}: {float(value):.6f}")

assert all(
    torch.isfinite(value)
    for value in losses.values()
)

print("Initial loss check passed.")

## 15. Train/validation helper

In [ ]:
def run_epoch(
    model,
    loader,
    beta,
    optimizer=None,
):
    training = optimizer is not None
    model.train(training)

    metric_names = [
        "total",
        "weighted_l1",
        "plain_l1",
        "chroma",
        "kl",
        "selection",
    ]

    sums = {name: 0.0 for name in metric_names}
    image_count = 0

    use_amp = USE_AMP and device.type == "cuda"

    progress = tqdm(
        loader,
        leave=False,
        desc="Train" if training else "Validation",
        mininterval=0.5,  # reduce tqdm overhead
    )

    for L, ab, _, _ in progress:
        # channels_last improves convolution throughput on A100.
        L = (
            L.to(device, non_blocking=True)
            .contiguous(memory_format=torch.channels_last)
        )
        ab = (
            ab.to(device, non_blocking=True)
            .contiguous(memory_format=torch.channels_last)
        )

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type="cuda",
                dtype=AMP_DTYPE,
                enabled=use_amp,
            ):
                outputs = model(L, ab)

                losses = cvae_v2_loss(
                    outputs[0],
                    ab,
                    outputs[1],
                    outputs[2],
                    outputs[3],
                    outputs[4],
                    beta,
                )

            if training:
                # BF16 on A100 has a large exponent range and does not need GradScaler.
                losses["total"].backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0,
                )

                optimizer.step()

        batch_size = L.size(0)

        for name in metric_names:
            sums[name] += losses[name].item() * batch_size

        image_count += batch_size

        progress.set_postfix(
            rec=f"{losses['plain_l1'].item():.4f}",
            kl=f"{losses['kl'].item():.4f}",
        )

    return {
        name: sums[name] / image_count
        for name in metric_names
    }

## 16. Train V2

The notebook writes checkpoints **directly to Google Drive**:

- `best_cvae_v2.pt`: best fixed validation selection score
- `last_cvae_v2.pt`: updated every epoch
- `epoch_005.pt`, `epoch_010.pt`, ...: milestone checkpoints

If Colab disconnects, rerunning the notebook with `RESUME_TRAINING=True` resumes from `last_cvae_v2.pt`.


In [ ]:
best_checkpoint_path = (
    MODEL_DIR
    / "best_cvae_v2.pt"
)

last_checkpoint_path = (
    MODEL_DIR
    / "last_cvae_v2.pt"
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

history = {
    "train_total": [],
    "train_plain_l1": [],
    "train_weighted_l1": [],
    "train_chroma": [],
    "train_kl": [],
    "val_total": [],
    "val_plain_l1": [],
    "val_weighted_l1": [],
    "val_chroma": [],
    "val_kl": [],
    "val_selection": [],
    "beta": [],
    "learning_rate": [],
    "epoch_seconds": [],
}

start_epoch = 0
best_selection = float("inf")
bad_epochs = 0

if (
    RESUME_TRAINING
    and last_checkpoint_path.exists()
):
    resume = torch.load(
        last_checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(
        resume["model_state_dict"]
    )

    optimizer.load_state_dict(
        resume["optimizer_state_dict"]
    )

    if "scheduler_state_dict" in resume:
        scheduler.load_state_dict(
            resume["scheduler_state_dict"]
        )

    start_epoch = int(
        resume["epoch"]
    )

    best_selection = float(
        resume["best_selection"]
    )

    bad_epochs = int(
        resume.get(
            "bad_epochs",
            0,
        )
    )

    history = resume["history"]

    print(
        "Resuming from epoch",
        start_epoch,
    )

else:
    print("Starting a new V2 training run.")


for epoch in range(
    start_epoch,
    EPOCHS,
):
    epoch_start = time.time()

    beta = beta_for_epoch(epoch)

    train_metrics = run_epoch(
        model,
        train_loader,
        beta,
        optimizer=optimizer,
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        beta,
        optimizer=None,
    )

    scheduler.step(
        val_metrics["selection"]
    )

    epoch_seconds = (
        time.time()
        - epoch_start
    )

    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )

    history["train_total"].append(
        train_metrics["total"]
    )
    history["train_plain_l1"].append(
        train_metrics["plain_l1"]
    )
    history["train_weighted_l1"].append(
        train_metrics["weighted_l1"]
    )
    history["train_chroma"].append(
        train_metrics["chroma"]
    )
    history["train_kl"].append(
        train_metrics["kl"]
    )

    history["val_total"].append(
        val_metrics["total"]
    )
    history["val_plain_l1"].append(
        val_metrics["plain_l1"]
    )
    history["val_weighted_l1"].append(
        val_metrics["weighted_l1"]
    )
    history["val_chroma"].append(
        val_metrics["chroma"]
    )
    history["val_kl"].append(
        val_metrics["kl"]
    )
    history["val_selection"].append(
        val_metrics["selection"]
    )

    history["beta"].append(beta)
    history["learning_rate"].append(
        current_lr
    )
    history["epoch_seconds"].append(
        epoch_seconds
    )

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"{epoch_seconds:5.1f}s | "
        f"beta={beta:.5f} | "
        f"train L1={train_metrics['plain_l1']:.4f} "
        f"KL={train_metrics['kl']:.4f} | "
        f"val L1={val_metrics['plain_l1']:.4f} "
        f"chroma={val_metrics['chroma']:.4f} "
        f"KL={val_metrics['kl']:.4f} | "
        f"select={val_metrics['selection']:.5f} | "
        f"lr={current_lr:.2e}"
    )

    improved = (
        val_metrics["selection"]
        < best_selection
    )

    if improved:
        best_selection = (
            val_metrics["selection"]
        )
        bad_epochs = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_selection": best_selection,
                "history": history,
                "config": config,
            },
            best_checkpoint_path,
        )

        print(
            "  ✓ saved new best model:",
            best_checkpoint_path,
        )

    else:
        bad_epochs += 1

    # Save a resumable checkpoint every epoch.
    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_selection": best_selection,
            "bad_epochs": bad_epochs,
            "history": history,
            "config": config,
        },
        last_checkpoint_path,
    )

    # Save milestones.
    if (
        (epoch + 1)
        % CHECKPOINT_EVERY
        == 0
    ):
        milestone_path = (
            MODEL_DIR
            / f"epoch_{epoch + 1:03d}.pt"
        )

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "best_selection": best_selection,
                "config": config,
            },
            milestone_path,
        )

        print(
            "  saved milestone:",
            milestone_path,
        )

    with open(
        METRIC_DIR
        / "training_history.json",
        "w",
    ) as file:
        json.dump(
            history,
            file,
            indent=2,
        )

    if (
        epoch + 1
        >= KL_WARMUP_EPOCHS
        and bad_epochs
        >= EARLY_STOPPING_PATIENCE
    ):
        print("Early stopping triggered.")
        break


print(
    "Best fixed validation score:",
    best_selection,
)

## 17. Load the best V2 model

In [ ]:
best_checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device,
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.eval()

history = best_checkpoint["history"]

print("Best epoch:", best_checkpoint["epoch"])
print(
    "Best selection score:",
    best_checkpoint["best_selection"],
)

## 18. Training curves — slide-ready

In [ ]:
epochs_axis = np.arange(
    1,
    len(history["val_selection"]) + 1,
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 8),
)

axes[0, 0].plot(
    epochs_axis,
    history["train_plain_l1"],
    label="Train",
)
axes[0, 0].plot(
    epochs_axis,
    history["val_plain_l1"],
    label="Validation",
)
axes[0, 0].set_title("Plain L1 reconstruction")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("L1")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.25)

axes[0, 1].plot(
    epochs_axis,
    history["train_chroma"],
    label="Train",
)
axes[0, 1].plot(
    epochs_axis,
    history["val_chroma"],
    label="Validation",
)
axes[0, 1].set_title("Chroma-magnitude loss")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Loss")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.25)

axes[1, 0].plot(
    epochs_axis,
    history["train_kl"],
    label="Train",
)
axes[1, 0].plot(
    epochs_axis,
    history["val_kl"],
    label="Validation",
)
axes[1, 0].set_title("Conditional KL")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("KL")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.25)

axes[1, 1].plot(
    epochs_axis,
    history["val_selection"],
    label="Fixed selection score",
)
axes[1, 1].set_title("Validation model-selection score")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Score")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.25)

fig.suptitle(
    "CVAE V2 Training Diagnostics",
    fontsize=16,
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "03_training_diagnostics.png",
    dpi=220,
    bbox_inches="tight",
)

plt.show()

## 19. Unified evaluation — the four group metrics

This function calculates:

### Normalized `ab` MSE

\[
\frac{1}{N}
\sum
(\widehat{ab}_{norm}-ab_{norm})^2
\]

### Normalized `ab` MAE

\[
\frac{1}{N}
\sum
|\widehat{ab}_{norm}-ab_{norm}|
\]

### RGB PSNR and RGB SSIM

Predicted normalized `ab` is converted back to LAB using

\[
ab=128\,ab_{norm},
\qquad
L=50(L_{norm}+1),
\]

and then LAB is converted to RGB. PSNR and SSIM are calculated per image and averaged.

For V2, the fair deterministic prediction uses the conditional prior mean \(z=\mu_p(L)\).

In [ ]:
@torch.inference_mode()
def evaluate_model(
    model,
    loader,
    mode,
    max_images=None,
    class_names=CIFAR10_CLASS_NAMES,
):
    model.eval()

    ab_squared_error_sum = 0.0
    ab_absolute_error_sum = 0.0
    ab_element_count = 0

    psnr_values = []
    ssim_values = []

    true_chroma_sum = 0.0
    pred_chroma_sum = 0.0
    chroma_element_count = 0

    per_class = defaultdict(
        lambda: {
            "count": 0,
            "ab_mse_sum": 0.0,
            "ab_mae_sum": 0.0,
            "ab_elements": 0,
            "psnr": [],
            "ssim": [],
        }
    )

    images_seen = 0

    for (
        L,
        true_ab,
        labels,
        _,
    ) in tqdm(
        loader,
        desc=f"Evaluating {mode}",
    ):
        L = (
            L.to(device, non_blocking=True)
            .contiguous(memory_format=torch.channels_last)
        )
        true_ab = (
            true_ab.to(device, non_blocking=True)
            .contiguous(memory_format=torch.channels_last)
        )

        if mode == "v2_prior_mean":
            pred_ab, _, _ = model.decode_prior(
                L,
                sample=False,
            )

        elif mode == "v1_z0":
            z = torch.zeros(
                L.size(0),
                model.latent_dim,
                device=device,
            )

            pred_ab = model.decode(
                L,
                z,
            )

        else:
            raise ValueError(
                f"Unknown evaluation mode: {mode}"
            )

        # ---------------------------
        # Normalized ab MSE + MAE
        # ---------------------------
        error = pred_ab - true_ab

        squared = error.pow(2)
        absolute = error.abs()

        ab_squared_error_sum += (
            squared.sum().item()
        )

        ab_absolute_error_sum += (
            absolute.sum().item()
        )

        ab_element_count += (
            true_ab.numel()
        )

        true_chroma = torch.sqrt(
            true_ab[:, 0].pow(2)
            + true_ab[:, 1].pow(2)
            + 1e-8
        )

        pred_chroma = torch.sqrt(
            pred_ab[:, 0].pow(2)
            + pred_ab[:, 1].pow(2)
            + 1e-8
        )

        true_chroma_sum += (
            true_chroma.sum().item()
        )
        pred_chroma_sum += (
            pred_chroma.sum().item()
        )
        chroma_element_count += (
            true_chroma.numel()
        )

        # ---------------------------
        # RGB PSNR + SSIM per image
        # ---------------------------
        for index in range(
            L.size(0)
        ):
            if (
                max_images is not None
                and images_seen >= max_images
            ):
                break

            true_rgb = normalized_lab_to_rgb(
                L[index],
                true_ab[index],
            )

            pred_rgb = normalized_lab_to_rgb(
                L[index],
                pred_ab[index],
            )

            psnr = peak_signal_noise_ratio(
                true_rgb,
                pred_rgb,
                data_range=1.0,
            )

            ssim = structural_similarity(
                true_rgb,
                pred_rgb,
                channel_axis=-1,
                data_range=1.0,
            )

            psnr_values.append(psnr)
            ssim_values.append(ssim)

            label = int(labels[index])

            image_sq = (
                squared[index]
                .sum()
                .item()
            )

            image_abs = (
                absolute[index]
                .sum()
                .item()
            )

            image_elements = (
                true_ab[index]
                .numel()
            )

            per_class[label]["count"] += 1
            per_class[label]["ab_mse_sum"] += image_sq
            per_class[label]["ab_mae_sum"] += image_abs
            per_class[label]["ab_elements"] += image_elements
            per_class[label]["psnr"].append(psnr)
            per_class[label]["ssim"].append(ssim)

            images_seen += 1

        if (
            max_images is not None
            and images_seen >= max_images
        ):
            break

    metrics = {
        "images": images_seen,
        "normalized_ab_mse": (
            ab_squared_error_sum
            / ab_element_count
        ),
        "normalized_ab_mae": (
            ab_absolute_error_sum
            / ab_element_count
        ),
        "rgb_psnr_db": float(
            np.mean(psnr_values)
        ),
        "rgb_ssim": float(
            np.mean(ssim_values)
        ),
        "true_mean_normalized_chroma": (
            true_chroma_sum
            / chroma_element_count
        ),
        "pred_mean_normalized_chroma": (
            pred_chroma_sum
            / chroma_element_count
        ),
    }

    metrics["chroma_retention_ratio"] = (
        metrics["pred_mean_normalized_chroma"]
        / metrics["true_mean_normalized_chroma"]
    )

    class_rows = []

    for class_id in sorted(
        per_class.keys()
    ):
        info = per_class[class_id]

        class_rows.append(
            {
                "class_id": class_id,
                "class_name": class_names[class_id],
                "images": info["count"],
                "normalized_ab_mse": (
                    info["ab_mse_sum"]
                    / info["ab_elements"]
                ),
                "normalized_ab_mae": (
                    info["ab_mae_sum"]
                    / info["ab_elements"]
                ),
                "rgb_psnr_db": float(
                    np.mean(info["psnr"])
                ),
                "rgb_ssim": float(
                    np.mean(info["ssim"])
                ),
            }
        )

    return (
        metrics,
        pd.DataFrame(class_rows),
    )

## 20. Evaluate V2 on the test set

In [ ]:
v2_metrics, v2_per_class = evaluate_model(
    model,
    test_loader,
    mode="v2_prior_mean",
    max_images=FINAL_METRIC_IMAGES,
)

print("CVAE V2 — conditional prior mean")
print(json.dumps(v2_metrics, indent=2))

display(
    pd.DataFrame(
        {
            "Test metric": [
                "Normalized ab MSE ↓",
                "Normalized ab MAE ↓",
                "RGB PSNR ↑",
                "RGB SSIM ↑",
                "Chroma retention",
            ],
            "CVAE V2": [
                v2_metrics["normalized_ab_mse"],
                v2_metrics["normalized_ab_mae"],
                f'{v2_metrics["rgb_psnr_db"]:.3f} dB',
                v2_metrics["rgb_ssim"],
                v2_metrics["chroma_retention_ratio"],
            ],
        }
    )
)

with open(
    METRIC_DIR
    / "cvae_v2_test_metrics.json",
    "w",
) as file:
    json.dump(
        v2_metrics,
        file,
        indent=2,
    )

v2_per_class.to_csv(
    METRIC_DIR
    / "cvae_v2_per_class_metrics.csv",
    index=False,
)

## 21. Evaluate the existing V1 checkpoint if available

The completed V1 notebook saved `best_cvae_cifar10.pt`. This cell searches the common Drive locations and, if found, calculates the **new normalized `ab` MSE and MAE** in addition to RGB PSNR and SSIM.

In [ ]:
v1_checkpoint_candidates = [
    DRIVE_DATA_DIR
    / "results"
    / "cvae_cifar10"
    / "best_cvae_cifar10.pt",

    DRIVE_DATA_DIR
    / "results"
    / "cvae_cifar10"
    / "cifar10_cvae_colorizer_final.pt",
]

V1_CHECKPOINT = None

for candidate in v1_checkpoint_candidates:
    if candidate.exists():
        V1_CHECKPOINT = candidate
        break

v1_metrics = None
v1_per_class = None
v1_model = None

if V1_CHECKPOINT is None:
    print(
        "V1 checkpoint was not found. "
        "V2 evaluation will continue normally."
    )

else:
    print("Loading V1:", V1_CHECKPOINT)

    v1_checkpoint = torch.load(
        V1_CHECKPOINT,
        map_location=device,
    )

    latent_dim_v1 = int(
        v1_checkpoint.get(
            "latent_dim",
            v1_checkpoint.get(
                "config",
                {},
            ).get(
                "latent_dim",
                64,
            ),
        )
    )

    v1_model = CVAEV1(
        latent_dim=latent_dim_v1
    ).to(device)

    v1_model.load_state_dict(
        v1_checkpoint[
            "model_state_dict"
        ]
    )

    v1_model.eval()

    (
        v1_metrics,
        v1_per_class,
    ) = evaluate_model(
        v1_model,
        test_loader,
        mode="v1_z0",
        max_images=FINAL_METRIC_IMAGES,
    )

    print(
        "CVAE V1 — z=0"
    )
    print(
        json.dumps(
            v1_metrics,
            indent=2,
        )
    )

    with open(
        METRIC_DIR
        / "cvae_v1_recalculated_metrics.json",
        "w",
    ) as file:
        json.dump(
            v1_metrics,
            file,
            indent=2,
        )

## 22. Compare against the CNN numbers from the supplied table

The following reference values are entered directly from the CNN numbers shared by my teammate:

| Metric | Tuned MSE model | L1 model |
|---|---:|---:|
| Normalized `ab` MSE ↓ | 0.008248 | 0.008492 |
| Normalized `ab` MAE ↓ | 0.061918 | 0.060978 |
| RGB PSNR ↑ | 25.225 dB | 25.440 dB |
| RGB SSIM ↑ | 0.938632 | 0.938432 |

The notebook appends the measured V1 and V2 results to the same table.

In [ ]:
comparison = {
    "Tuned MSE model": {
        "Normalized ab MSE ↓": 0.008248,
        "Normalized ab MAE ↓": 0.061918,
        "RGB PSNR ↑": 25.225,
        "RGB SSIM ↑": 0.938632,
    },
    "L1 model": {
        "Normalized ab MSE ↓": 0.008492,
        "Normalized ab MAE ↓": 0.060978,
        "RGB PSNR ↑": 25.440,
        "RGB SSIM ↑": 0.938432,
    },
}

if v1_metrics is not None:
    comparison["CVAE V1"] = {
        "Normalized ab MSE ↓": v1_metrics["normalized_ab_mse"],
        "Normalized ab MAE ↓": v1_metrics["normalized_ab_mae"],
        "RGB PSNR ↑": v1_metrics["rgb_psnr_db"],
        "RGB SSIM ↑": v1_metrics["rgb_ssim"],
    }

comparison["CVAE V2"] = {
    "Normalized ab MSE ↓": v2_metrics["normalized_ab_mse"],
    "Normalized ab MAE ↓": v2_metrics["normalized_ab_mae"],
    "RGB PSNR ↑": v2_metrics["rgb_psnr_db"],
    "RGB SSIM ↑": v2_metrics["rgb_ssim"],
}

comparison_df = pd.DataFrame(
    comparison
)

display(
    comparison_df.style.format(
        {
            column: "{:.6f}"
            for column in comparison_df.columns
        }
    )
)

comparison_df.to_csv(
    METRIC_DIR
    / "model_metric_comparison.csv"
)

## 23. Metric comparison figure — slide-ready

In [ ]:
metric_names = list(
    comparison_df.index
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 8),
)

for axis, metric_name in zip(
    axes.flatten(),
    metric_names,
):
    values = (
        comparison_df
        .loc[metric_name]
    )

    axis.bar(
        values.index,
        values.values,
    )

    axis.set_title(metric_name)
    axis.tick_params(
        axis="x",
        rotation=20,
    )
    axis.grid(
        axis="y",
        alpha=0.25,
    )

fig.suptitle(
    "CIFAR-10 Colorization — Test Metric Comparison",
    fontsize=16,
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "04_model_metric_comparison.png",
    dpi=220,
    bbox_inches="tight",
)

plt.show()

## 24. Select one test example from every CIFAR-10 class

In [ ]:
def one_index_per_class(dataset):
    found = {}

    for index in range(len(dataset)):
        _, _, label, _ = dataset[index]

        if label not in found:
            found[label] = index

        if len(found) == 10:
            break

    return [
        found[class_id]
        for class_id in range(10)
    ]


presentation_indices = one_index_per_class(
    test_dataset
)

print(
    dict(
        zip(
            CIFAR10_CLASS_NAMES,
            presentation_indices,
        )
    )
)

## 25. V2 qualitative results — one example per class

In [ ]:
@torch.no_grad()
def create_v2_qualitative_grid(
    indices,
):
    model.eval()

    columns = len(indices)

    fig, axes = plt.subplots(
        3,
        columns,
        figsize=(2.0 * columns, 6.2),
    )

    for column, dataset_index in enumerate(indices):
        L, true_ab, label, _ = test_dataset[
            dataset_index
        ]

        L_device = (
            L.unsqueeze(0)
            .to(device)
        )

        pred_ab, _, _ = model.decode_prior(
            L_device,
            sample=False,
        )

        pred_ab = pred_ab.squeeze(0)

        axes[0, column].imshow(
            normalized_L_to_gray(L),
            cmap="gray",
            vmin=0,
            vmax=1,
        )
        axes[0, column].set_title(
            CIFAR10_CLASS_NAMES[label]
        )
        axes[0, column].axis("off")

        axes[1, column].imshow(
            normalized_lab_to_rgb(
                L,
                true_ab,
            )
        )
        axes[1, column].axis("off")

        axes[2, column].imshow(
            normalized_lab_to_rgb(
                L,
                pred_ab,
            )
        )
        axes[2, column].axis("off")

    axes[0, 0].set_ylabel("Grayscale")
    axes[1, 0].set_ylabel("Ground truth")
    axes[2, 0].set_ylabel("CVAE V2")

    fig.suptitle(
        "CVAE V2 Deterministic Colorization — Conditional Prior Mean",
        fontsize=16,
    )

    plt.tight_layout()

    output_path = (
        FIGURE_DIR
        / "05_v2_qualitative_one_per_class.png"
    )

    fig.savefig(
        output_path,
        dpi=220,
        bbox_inches="tight",
    )

    plt.show()

    return output_path


qualitative_path = create_v2_qualitative_grid(
    presentation_indices
)

print("Saved:", qualitative_path)

## 26. V1 versus V2 qualitative comparison

In [ ]:
@torch.no_grad()
def create_v1_v2_comparison(
    indices,
):
    if v1_model is None:
        print(
            "Skipping V1 vs V2 figure because "
            "the V1 checkpoint was not found."
        )
        return None

    v1_model.eval()
    model.eval()

    columns = len(indices)

    fig, axes = plt.subplots(
        4,
        columns,
        figsize=(2.0 * columns, 8.2),
    )

    for column, dataset_index in enumerate(indices):
        L, true_ab, label, _ = test_dataset[
            dataset_index
        ]

        L_device = (
            L.unsqueeze(0)
            .to(device)
        )

        z0 = torch.zeros(
            1,
            v1_model.latent_dim,
            device=device,
        )

        v1_ab = (
            v1_model
            .decode(
                L_device,
                z0,
            )
            .squeeze(0)
        )

        v2_ab, _, _ = model.decode_prior(
            L_device,
            sample=False,
        )

        v2_ab = v2_ab.squeeze(0)

        axes[0, column].imshow(
            normalized_L_to_gray(L),
            cmap="gray",
            vmin=0,
            vmax=1,
        )
        axes[0, column].set_title(
            CIFAR10_CLASS_NAMES[label]
        )
        axes[0, column].axis("off")

        axes[1, column].imshow(
            normalized_lab_to_rgb(
                L,
                true_ab,
            )
        )
        axes[1, column].axis("off")

        axes[2, column].imshow(
            normalized_lab_to_rgb(
                L,
                v1_ab,
            )
        )
        axes[2, column].axis("off")

        axes[3, column].imshow(
            normalized_lab_to_rgb(
                L,
                v2_ab,
            )
        )
        axes[3, column].axis("off")

    axes[0, 0].set_ylabel("Gray")
    axes[1, 0].set_ylabel("Truth")
    axes[2, 0].set_ylabel("V1")
    axes[3, 0].set_ylabel("V2")

    fig.suptitle(
        "CVAE V1 vs V2 Colorization",
        fontsize=16,
    )

    plt.tight_layout()

    output_path = (
        FIGURE_DIR
        / "06_v1_vs_v2_qualitative.png"
    )

    fig.savefig(
        output_path,
        dpi=220,
        bbox_inches="tight",
    )

    plt.show()

    return output_path


create_v1_v2_comparison(
    presentation_indices
)

## 27. Multiple plausible V2 colorizations

Unlike a deterministic CNN, the CVAE can sample several colour hypotheses from

\[
p_\psi(z\mid L).
\]


In [ ]:
@torch.no_grad()
def create_diversity_figure(
    dataset_index,
    num_samples=8,
    temperature=1.0,
):
    model.eval()

    L, true_ab, label, _ = test_dataset[
        dataset_index
    ]

    L_device = (
        L.unsqueeze(0)
        .to(device)
    )

    fig, axes = plt.subplots(
        2,
        5,
        figsize=(13, 5.5),
    )

    axes = axes.flatten()

    axes[0].imshow(
        normalized_L_to_gray(L),
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[0].set_title("Grayscale")
    axes[0].axis("off")

    axes[1].imshow(
        normalized_lab_to_rgb(
            L,
            true_ab,
        )
    )
    axes[1].set_title("Ground truth")
    axes[1].axis("off")

    prior_mean, _, _ = model.decode_prior(
        L_device,
        sample=False,
    )

    axes[2].imshow(
        normalized_lab_to_rgb(
            L,
            prior_mean.squeeze(0),
        )
    )
    axes[2].set_title("Prior mean")
    axes[2].axis("off")

    for sample_number in range(
        min(
            num_samples,
            len(axes) - 3,
        )
    ):
        sampled_ab, _, _ = model.decode_prior(
            L_device,
            sample=True,
            temperature=temperature,
        )

        axis = axes[
            sample_number + 3
        ]

        axis.imshow(
            normalized_lab_to_rgb(
                L,
                sampled_ab.squeeze(0),
            )
        )
        axis.set_title(
            f"Sample {sample_number + 1}"
        )
        axis.axis("off")

    for index in range(
        3 + min(num_samples, len(axes) - 3),
        len(axes),
    ):
        axes[index].axis("off")

    fig.suptitle(
        "Conditional VAE Diversity — "
        + CIFAR10_CLASS_NAMES[label],
        fontsize=16,
    )

    plt.tight_layout()

    output_path = (
        FIGURE_DIR
        / "07_v2_multiple_colorizations.png"
    )

    fig.savefig(
        output_path,
        dpi=220,
        bbox_inches="tight",
    )

    plt.show()

    return output_path


# Change this index if another example looks better for the slide.
create_diversity_figure(
    presentation_indices[1],
    num_samples=7,
    temperature=1.0,
)

## 28. Chroma-retention diagnostic

In [ ]:
chroma_rows = [
    {
        "Model": "Ground truth",
        "Mean normalized chroma": (
            v2_metrics[
                "true_mean_normalized_chroma"
            ]
        ),
    },
    {
        "Model": "CVAE V2",
        "Mean normalized chroma": (
            v2_metrics[
                "pred_mean_normalized_chroma"
            ]
        ),
    },
]

if v1_metrics is not None:
    chroma_rows.insert(
        1,
        {
            "Model": "CVAE V1",
            "Mean normalized chroma": (
                v1_metrics[
                    "pred_mean_normalized_chroma"
                ]
            ),
        },
    )

chroma_df = pd.DataFrame(
    chroma_rows
)

display(chroma_df)

fig = plt.figure(
    figsize=(7, 4.5)
)

plt.bar(
    chroma_df["Model"],
    chroma_df["Mean normalized chroma"],
)

plt.ylabel(
    "Mean normalized chroma magnitude"
)
plt.title(
    "Color Saturation / Chroma Retention"
)
plt.grid(
    axis="y",
    alpha=0.25,
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "08_chroma_retention.png",
    dpi=220,
    bbox_inches="tight",
)

plt.show()

## 29. Per-class performance

In [ ]:
display(
    v2_per_class[
        [
            "class_name",
            "normalized_ab_mse",
            "normalized_ab_mae",
            "rgb_psnr_db",
            "rgb_ssim",
        ]
    ].sort_values(
        "rgb_psnr_db",
        ascending=False,
    )
)

fig = plt.figure(
    figsize=(11, 5)
)

plt.bar(
    v2_per_class["class_name"],
    v2_per_class["rgb_psnr_db"],
)

plt.ylabel("RGB PSNR (dB)")
plt.xlabel("CIFAR-10 class")
plt.title(
    "CVAE V2 Test PSNR by CIFAR-10 Class"
)
plt.xticks(
    rotation=35,
    ha="right",
)
plt.grid(
    axis="y",
    alpha=0.25,
)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "09_v2_per_class_psnr.png",
    dpi=220,
    bbox_inches="tight",
)

plt.show()

## 30. Latent-space PCA

In [ ]:
@torch.no_grad()
def collect_posterior_latents(
    model,
    loader,
    max_images=3000,
):
    model.eval()

    mus = []
    labels_all = []
    seen = 0

    for L, ab, labels, _ in tqdm(
        loader,
        desc="Collecting latents",
    ):
        L = L.to(device)
        ab = ab.to(device)

        mu_q, _ = model.encode_posterior(
            L,
            ab,
        )

        remaining = (
            max_images - seen
        )

        take = min(
            remaining,
            L.size(0),
        )

        mus.append(
            mu_q[:take]
            .detach()
            .cpu()
            .numpy()
        )

        labels_all.append(
            labels[:take]
            .cpu()
            .numpy()
        )

        seen += take

        if seen >= max_images:
            break

    return (
        np.concatenate(mus),
        np.concatenate(labels_all),
    )


latent_mu, latent_labels = collect_posterior_latents(
    model,
    test_loader,
    max_images=3000,
)

pca = PCA(
    n_components=2,
    random_state=SEED,
)

latent_2d = pca.fit_transform(
    latent_mu
)

fig = plt.figure(
    figsize=(9, 7)
)

for class_id in range(10):
    mask = (
        latent_labels
        == class_id
    )

    plt.scatter(
        latent_2d[mask, 0],
        latent_2d[mask, 1],
        s=9,
        alpha=0.6,
        label=CIFAR10_CLASS_NAMES[class_id],
    )

plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title(
    "CVAE V2 Posterior Latent Space "
    f"({pca.explained_variance_ratio_.sum() * 100:.1f}% variance in PC1+PC2)"
)
plt.legend(
    ncol=2,
    fontsize=8,
)
plt.grid(alpha=0.25)

plt.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "10_v2_latent_pca.png",
    dpi=220,
    bbox_inches="tight",
)

plt.show()

print(
    "Explained variance:",
    pca.explained_variance_ratio_,
)

## 31. Save final inference model

In [ ]:
final_model_path = (
    MODEL_DIR
    / "cvae_v2_final_inference.pt"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": config,
        "best_epoch": best_checkpoint["epoch"],
        "best_selection": best_checkpoint[
            "best_selection"
        ],
        "test_metrics": v2_metrics,
    },
    final_model_path,
)

print("Saved:", final_model_path)

## 32. Create a summary table

In [ ]:
summary_rows = [
    {
        "Item": "Dataset",
        "Result": (
            "CIFAR-10: 45,000 train / "
            "5,000 validation / 10,000 test"
        ),
    },
    {
        "Item": "Input / target",
        "Result": (
            "LAB L channel → LAB ab channels"
        ),
    },
    {
        "Item": "VAE type",
        "Result": (
            "Conditional VAE with p(z|L) "
            "and q(z|L,ab)"
        ),
    },
    {
        "Item": "Latent dimension",
        "Result": LATENT_DIM,
    },
    {
        "Item": "Best epoch",
        "Result": best_checkpoint["epoch"],
    },
    {
        "Item": "Normalized ab MSE ↓",
        "Result": (
            f"{v2_metrics['normalized_ab_mse']:.6f}"
        ),
    },
    {
        "Item": "Normalized ab MAE ↓",
        "Result": (
            f"{v2_metrics['normalized_ab_mae']:.6f}"
        ),
    },
    {
        "Item": "RGB PSNR ↑",
        "Result": (
            f"{v2_metrics['rgb_psnr_db']:.3f} dB"
        ),
    },
    {
        "Item": "RGB SSIM ↑",
        "Result": (
            f"{v2_metrics['rgb_ssim']:.6f}"
        ),
    },
    {
        "Item": "Chroma retention",
        "Result": (
            f"{100 * v2_metrics['chroma_retention_ratio']:.1f}%"
        ),
    },
]

summary_df = pd.DataFrame(
    summary_rows
)

display(summary_df)


## 33. Final output inventory

In [ ]:
print("RESULTS SAVED TO:")
print(DRIVE_OUTPUT_DIR)

print("\nMODELS")
for path in sorted(MODEL_DIR.glob("*")):
    print(
        f"  {path.name:35s} "
        f"{path.stat().st_size / 1024**2:8.2f} MB"
    )

print("\nMETRICS")
for path in sorted(METRIC_DIR.glob("*")):
    print(" ", path.name)
